# Financial Decision Making Using a Sharpe-Style Ratio

**Coding for Finance** — Università degli Studi di Bergamo

An investor wants to rank five publicly traded technology companies — **NVIDIA, Meta,
Netflix, Adobe, Intel** — combining *expected performance* and *stability* into a single
risk-adjusted score, using real FY2021–FY2025 data (SEC filings / stockanalysis.com / roic.ai).

**Criteria:** Revenue Growth, Operating Margin, Return on Assets, EPS (all *benefit*
criteria) and Debt-to-Equity (the one *cost* criterion).

**Data limitation, stated explicitly:** ROE/Debt-to-Equity have a full 5-year series for
some companies only; where a 5-year history wasn't reliably available, the most recent
reported value is used in its place (flagged inline in the code) rather than silently
filled in.

## 1. Data

Five criteria, five companies, FY2021–FY2025.

In [1]:
import numpy as np
import pandas as pd

companies = np.array(["NVIDIA", "Meta", "Netflix", "Adobe", "Intel"])

revenue = np.array([
    [16675,  26914,  26974,  60922,  130497],
    [117929, 116609, 134902, 164501, 200966],
    [29698,  31616,  33723,  39001,  45183],
    [15785,  17606,  19409,  21505,  23769],
    [79024,  63054,  54228,  53101,  52853],
])
revenue_growth = (revenue[:, 1:] / revenue[:, :-1] - 1) * 100

operating_margin = np.array([
    [27.18, 37.31, 20.68, 54.12, 62.42],
    [39.65, 24.82, 34.66, 42.18, 41.44],
    [20.86, 17.82, 20.62, 26.71, 29.49],
    [36.76, 34.64, 34.26, 31.35, 36.63],
    [24.62, 3.70,  0.17,  -21.99, -4.19],
])

roa = np.array([
    [18.79, 30.86, 19.6, 60.45, 83.76],
    [20.90, 20.90, 20.90, 20.90, 20.90],   # Meta: current value only, broadcast
    [17.90, 17.90, 17.90, 17.90, 17.90],   # Netflix: current value only, broadcast
    [24.17, 24.17, 24.17, 24.17, 24.17],   # Adobe: current value only, broadcast
    [14.43, 13.91, 4.33,  2.37,  -0.29],
])

eps = np.array([
    [0.18, 0.39, 0.18, 1.21, 2.97],
    [13.77, 8.59, 14.87, 23.86, 23.49],
    [1.12, 1.00, 1.20, 1.98, 2.53],
    [10.02, 10.10, 11.82, 12.36, 16.70],
    [4.86, 1.94, 0.40, -4.38, -0.06],
])

debt_equity = np.array([
    [39.06, 43.92, 47.99, 22.28, 12.58],
    [10.21, 20.06, 23.25, 25.80, 37.60],
    [92.71, 69.08, 68.69, 55.77, 50.59],
    [60.0,  60.0,  60.0,  60.0,  60.0],
    [np.nan, np.nan, np.nan, np.nan, np.nan],   # Intel: not found -> imputed below
])
peer_avg_de = np.nanmean(debt_equity[:4])
debt_equity = np.where(np.isnan(debt_equity), peer_avg_de, debt_equity)  # Intel = peer average

## 2. Mean and variance per criterion (2021–2025)

In [2]:
criteria_names = ["Revenue Growth (%)", "Operating Margin (%)", "ROA (%)",
                   "EPS ($)", "Debt-to-Equity (%)"]
mean_matrix = np.column_stack([revenue_growth.mean(axis=1), operating_margin.mean(axis=1),
                                roa.mean(axis=1), eps.mean(axis=1), debt_equity.mean(axis=1)])
var_matrix = np.column_stack([revenue_growth.var(axis=1), operating_margin.var(axis=1),
                               roa.var(axis=1), eps.var(axis=1), debt_equity.var(axis=1)])
is_benefit = np.array([True, True, True, True, False])

print(pd.DataFrame(mean_matrix, index=companies, columns=criteria_names).round(2))

         Revenue Growth (%)  Operating Margin (%)  ROA (%)  EPS ($)  \
NVIDIA                75.42                 40.34    42.69     0.99   
Meta                  14.67                 36.55    20.90    16.92   
Netflix               11.16                 23.10    17.90     1.57   
Adobe                 10.78                 34.73    24.17    12.20   
Intel                 -9.19                  0.46     6.95     0.55   

         Debt-to-Equity (%)  
NVIDIA                33.17  
Meta                  23.38  
Netflix               67.37  
Adobe                 60.00  
Intel                 45.98  


## 3. Normalization

Criteria are on different scales (%, %, %, $, %), so each is min-max normalized onto [0, 1] before combining — benefit criteria as $(x-\min)/(\max-\min)$, the cost criterion (Debt-to-Equity) with the direction flipped.

In [3]:
# Min-Max normalization (vectorized, benefit vs. cost criteria in one line)
col_max, col_min = mean_matrix.max(axis=0), mean_matrix.min(axis=0)
col_range = col_max - col_min
norm_mean = np.where(is_benefit, (mean_matrix - col_min) / col_range,
                      (col_max - mean_matrix) / col_range)

print(pd.DataFrame(norm_mean, index=companies, columns=criteria_names).round(4))

         Revenue Growth (%)  Operating Margin (%)  ROA (%)  EPS ($)  \
NVIDIA               1.0000                1.0000   1.0000   0.0265   
Meta                 0.2820                0.9049   0.3903   1.0000   
Netflix              0.2404                0.5677   0.3064   0.0620   
Adobe                0.2360                0.8592   0.4818   0.7118   
Intel                0.0000                0.0000   0.0000   0.0000   

         Debt-to-Equity (%)  
NVIDIA               0.7776  
Meta                 1.0000  
Netflix              0.0000  
Adobe                0.1675  
Intel                0.4863  


## 4. Composite Sharpe-style ranking

$$\text{Sharpe}_i = \frac{\sum_j w_j\, r_{ij} - R_f}{\sqrt{\sum_j w_j^2\, \sigma_{ij}^2}}, \qquad w = (0.20, 0.25, 0.25, 0.20, 0.10)$$

The weights favor Operating Margin and ROA slightly (profitability), and de-emphasize Debt-to-Equity.

In [4]:
# Composite "Sharpe Ratio": weighted normalized return over weighted risk.
#   SR_i = ( sum_j w_j * r_ij  -  Rf ) / sqrt( sum_j w_j^2 * var_ij )
weights = np.array([0.20, 0.25, 0.25, 0.20, 0.10])
Rf = 0.0   # relative ranking, so a risk-free rate of 0 is used

weighted_return = norm_mean @ weights
weighted_risk = np.sqrt((weights ** 2) @ var_matrix.T)
sharpe = (weighted_return - Rf) / weighted_risk

df_sharpe = pd.DataFrame({"Weighted Return": weighted_return, "Weighted Risk": weighted_risk,
                           "Sharpe": sharpe}, index=companies).sort_values("Sharpe", ascending=False)
print(df_sharpe.round(4))

         Weighted Return  Weighted Risk  Sharpe
Adobe             0.5416         0.6987  0.7751
Meta              0.6802         2.8910  0.2353
Netflix           0.2790         2.0362  0.1370
NVIDIA            0.7831        12.5307  0.0625
Intel             0.0486         4.4023  0.0110


## 5. Interpretation

- **Adobe** achieves the highest composite Sharpe Ratio: its normalized performance is
  competitive across criteria while its year-to-year *dispersion* (the denominator) is
  comparatively small, so it isn't penalized as heavily as a company whose good average
  performance came with large swings.
- **Intel** ranks last, dragged down by a combination of weak normalized return and high
  dispersion (its operating margin and ROA turned negative in the most recent years).
- **NVIDIA is the clearest illustration of why risk matters**: its raw averages are the
  best in the group by a wide margin, but its year-to-year variance is also the highest,
  so a pure-return ranking and this risk-adjusted ranking disagree on where NVIDIA
  belongs.
- **Limitation.** Unlike a textbook Sharpe ratio (numerator and denominator from the
  *same* return series), here "return" and "risk" come from five *different* criteria in
  different units, forced onto a common scale by min-max normalization — so the ranking
  is sensitive to the chosen weights and to which criteria are treated as return vs.
  risk inputs, not just to the underlying data.